**Eye-Tracking & Neurocognitive Intelligence | Reading Comprehension & Neurocognitive Modelling | Presented by: Reza Saadatyar (2026)**

**02 | Reading Behaviour and Statistical Analysis**

![Reading Behaviour and Statistical Analysis — scientific workflow](../../images/02_reading_behaviour_and_statistics.svg)

**Objective:** Describe pre-answer eye movements, derive trial features and ordered sequences, and examine exploratory associations with response correctness.

**Dataset:** Matched eligible paragraph trials and fixed participant assignments from Notebook 01.

**Scope:** Construct scalar gaze and paragraph features, aligned fixation sequences and exploratory Training-only associations.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("reading_comprehension_modelling")

# ---------------------------- 2. Standard Library ----------------------------
import subprocess
import sys

# --------------------------- 3. Study Dependencies ---------------------------
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(root / "reading_comprehension_modelling" / "requirements.txt")],
    check=True,
)

# ----------------- 4. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 5. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

# ----------------- 6. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from onestop_io import (
    acquire_reports, read_report, make_cohort,
    validate_splits, validate_trial_index,
)
from reading_features import build_features, GAZE, TEXT
from reading_statistics import reading_associations

# ------------------- 7. Output Directory and Setup Summary -------------------
out_dir = root / "reading_comprehension_modelling" / "output"
out_dir.mkdir(parents=True, exist_ok=True)
raw_dir = root / "data" / "raw" / "onestop" / "ordinary"
processed = root / "data" / "processed" / "onestop"
processed.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4),
)
print("Study outputs:", out_dir)
print("Source directory:", raw_dir)
print("Local data directory:", processed)
print("Python:", sys.version.split()[0])


**02. Load the Fixed Cohort and Paragraph Reports**

Use the exact eligible trial identities and participant roles from Notebook 01. Reuse the original paragraph reports; no re-sampling or new split is performed.

In [ ]:
# ============== 02. Load the Fixed Cohort and Paragraph Reports ==============

# ------------------- 1. Load Trial and Participant Tables --------------------
trials = pd.read_csv(out_dir/"01_trial_index.csv",dtype=str)
trials["is_correct"] = trials.is_correct.astype(int)
splits = pd.read_csv(out_dir/"01_participant_splits.csv",dtype=str)
validate_splits(trials,splits)

# -------------------- 2. Read Original Paragraph Reports ---------------------
reports = acquire_reports(raw_dir)
fix = read_report(reports["fixations"],"fixations")
ia = read_report(reports["ia"],"ia")

# ------------------- 3. Reconcile Source Trial Identities --------------------
current, _ = make_cohort(fix,ia)
validate_trial_index(trials,current)
del current


**03. Extract Reading and Text Features**

Aggregate valid fixations in milliseconds and screen pixels. Backward word-index transitions are a proxy for regressions, not a validated saccade detector. Word coverage and dwell time come from interest areas. Text predictors are derived from the whole paragraph, not from fixated words. Source frequency and surprisal summaries are descriptive only because report coverage and layout may vary.

In [ ]:
# =================== 03. Extract Reading and Text Features ===================

# ------------------ 1. Compute Trial Features and Sequences ------------------
table, x, mask = build_features(trials,fix,ia,max_fixations=256)
validate_splits(table,splits)

# ----------------------- 2. Align Whole-Paragraph Text -----------------------
paragraph_text = (ia[["participant_id","trial_index","paragraph"]].drop_duplicates()
                  .merge(trials[["participant_id","trial_index","trial_id"]],
                         on=["participant_id","trial_index"],validate="one_to_one")
                  [["trial_id","paragraph"]])
if len(paragraph_text)!=len(table):
    raise ValueError("Missing paragraph text")

# ----------------- 3. Describe Feature and Sequence Support ------------------
print("Feature rows:",len(table))
print("Sequence shape:",x.shape)
print("Truncated sequences:",int(table.sequence_truncated.sum()))
display(table[GAZE+TEXT].describe().T)
del fix, ia


**04. Describe Reading Behaviour on Training Trials**

Exploratory associations and visualisations use only Protocol A Training participants. Repeated trials within the same person are not independent observations. Report participant-balanced summaries by outcome; a participant without one outcome does not contribute to that outcome's mean.

In [ ]:
# ============= 04. Describe Reading Behaviour on Training Trials =============

# --------------------- 1. Identify Training Participants ---------------------
train_ids = splits.loc[splits.protocol.eq("A_unseen_people") &
                       splits.role.eq("train"),"participant_id"]
train = table[table.participant_id.isin(train_ids)].copy()

# ---------------- 2. Summarise Participant Reading Behaviour -----------------
person = train.groupby(["participant_id","is_correct"])[GAZE].mean().reset_index()
summary = person.groupby("is_correct")[GAZE].agg(["mean","std","count"])
summary.columns = ["_".join(c) for c in summary.columns]
summary = summary.reset_index()
display(summary.T)

# ------------------- 3. Plot Participant Fixation Duration -------------------
fig, ax = plt.subplots(figsize=(8,4))
person.boxplot(column="fix_mean_ms",by="is_correct",ax=ax)
fig.suptitle("")
ax.set(title="Training participant means",xlabel="Correct response (0 / 1)",ylabel="Mean fixation duration (ms)")
fig.tight_layout()
save_figure(fig,out_dir/"02_reading_behaviour.png")
plt.show()


**05. Estimate Crossed Participant and Paragraph Associations**

Fit separate exploratory Bayesian logistic mixed-effects models with participant and paragraph random intercepts and difficulty as a fixed adjustment. Standardise each feature on the analysed Training rows. Report variational posterior means, approximate intervals and optimiser status, not causal effects or confirmatory significance tests.

In [ ]:
# ======== 05. Estimate Crossed Participant and Paragraph Associations ========

# ------------------- 1. Estimate Exploratory Associations --------------------
associations = reading_associations(train)
display(associations)
if associations.empty:
    raise ValueError("No estimable exploratory associations")

# ------------------ 2. Plot Approximate Posterior Intervals ------------------
fig, ax = plt.subplots(figsize=(9,5))
pos = np.arange(len(associations))
ax.hlines(pos,associations.approx_low,associations.approx_high)
ax.scatter(associations.log_odds_per_sd,pos)
ax.axvline(0,color="grey",linestyle="--")
ax.set_yticks(pos,associations.feature)
ax.set(xlabel="Log odds per SD (approximate posterior interval)",title="Exploratory Training-only associations")
fig.tight_layout()
save_figure(fig,out_dir/"02_reading_associations.png")
plt.show()


**06. Save Feature Tables and Local Sequence Arrays**

Store one feature row per eligible trial and a fixed-length sequence with an explicit valid-position mask. Sequence truncation is recorded; local raw-text and NPZ files are not committed. The numerical table is reused by both modelling notebooks.

In [ ]:
# ============= 06. Save Feature Tables and Local Sequence Arrays =============

# ---------------------- 1. Save Derived Feature Tables -----------------------
table.to_csv(out_dir/"02_trial_features.csv",index=False)
summary.to_csv(out_dir/"02_training_behaviour.csv",index=False)
associations.to_csv(out_dir/"02_training_associations.csv",index=False)

# ------------------ 2. Save Local Text and Sequence Arrays -------------------
paragraph_text.to_csv(processed/"02_paragraph_text.csv",index=False)
np.savez_compressed(processed/"02_fixation_sequences.npz",
                    trial_id=table.trial_id.to_numpy(str),x=x,mask=mask)

# ------------------------ 3. Package Current Outputs -------------------------
download_outputs_from_colab(out_dir,"02")
